In [1]:
from pathlib import Path
import shutil

project_dir = Path.cwd().parent
input_dir = project_dir / 'input' / 'datasets' / 'b1leygr'
(working_dir := project_dir / 'working/models/rfdetr_medium').mkdir(parents=True, exist_ok=True)
shutil.copy(input_dir / 'lastckpt' / 'last.ckpt', working_dir)
project_dir, input_dir, working_dir

(PosixPath('/kaggle'),
 PosixPath('/kaggle/input/datasets/b1leygr'),
 PosixPath('/kaggle/working/models/rfdetr_medium'))

In [2]:
!nvidia-smi

Sun Jun  7 14:11:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%pip install -q rfdetr "rfdetr[loggers]" supervision torch faster-coco-eval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [4]:
from rfdetr import RFDETRMedium
import torch

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Memory Cached: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

Using device: cuda
GPU Name: Tesla T4
Memory Allocated: 0.00 GB
Memory Cached: 0.00 GB


In [6]:
model = RFDETRMedium(num_classes=12)

model.train(
    dataset_dir=str(input_dir / 'chessman-detection' / 'Chessman Detection.v5i.coco'),
    output_dir=str(working_dir),
    resume=str(working_dir / 'last.ckpt'),
    epochs=48,
    resolution=576,
    aug_config={
        "HorizontalFlip": {"p": 0.5}
    },
    checkpoint_interval=5,
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    lr_scheduler='step',
    tensorboard=True,
    device=device
)

[2026-06-07 14:12:22] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-medium.pth


/root/.roboflow/models/rf-detr-medium.pth:   0%|          | 0.00/386M [00:00<?, ?iB/s]

[2026-06-07 14:12:33] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-medium.pth


[2026-06-07 14:12:33] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-07 14:12:33] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-06-07 14:12:34] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


...

Restored all states from the checkpoint at /kaggle/working/models/rfdetr_medium/last.ckpt
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Output()

[2026-06-07 15:36:08] [INFO] rf-detr - Best EMA mAP improved to 0.6989 (epoch 32)
[2026-06-07 16:30:53] [INFO] rf-detr - Best EMA mAP improved to 0.6994 (epoch 34)
[2026-06-07 17:27:05] [INFO] rf-detr - Best regular mAP saved to /kaggle/working/models/rfdetr_medium/checkpoint_best_regular.pth (epoch 36)
[2026-06-07 18:49:55] [INFO] rf-detr - Best EMA mAP improved to 0.6997 (epoch 39)


`Trainer.fit` stopped: `max_epochs=48` reached.


[2026-06-07 22:29:31] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.6945, ema=0.6997)
